# Background to signal ST tags
## Calculate background to signal ratios single tags

### Include library for handling uncertainties
#### [Here is the ```uncertainties-cpp``` library on GitHub](https://github.com/Gattocrucco/uncertainties-cpp)

In [1]:
gInterpreter->AddIncludePath("/data/lhcb/users/tat/uncertainties-cpp");

In [2]:
#include<uncertainties/impl.hpp>
#include<uncertainties/ureal.hpp>
#include<uncertainties/io.hpp>
#include<uncertainties/math.hpp>

### Load utility functions

In [3]:
gROOT->ProcessLine(".L ../UtilityFunctions.C");

### Get reconstructed background efficiency

In [4]:
uncertainties::udouble GetRecBackgroundEff(const std::string &TagMode,
                                          const std::string &BackgroundMode) {
    std::string Filename = "${BES3_ANALYSIS_PATH}/Selection/PeakingBackgrounds/SingleTag/";
    Filename += TagMode + "/" + BackgroundMode + "_to_" + TagMode + "_SingleTag_SignalMC.root";
    TChain Chain((TagMode + "SingleTag").c_str());
    Chain.Add(Filename.c_str());
    const double Yield = Chain.GetEntries();
    const double Eff = Yield/400000.0;
    const double Eff_err = TMath::Sqrt(Eff*(1.0 - Eff)/400000.0);
    return uncertainties::udouble(Eff, Eff_err);
}

### List of tags and their backgrounds

In [5]:
const std::map<std::string, std::vector<std::string>> Tags{
    {"KKpipi", std::vector<std::string>{"KSKK"}},
    {"Kpipipi", std::vector<std::string>{"KSKpi"}},
    {"KSpi0", std::vector<std::string>{"pipipi0"}},
    {"pipipi0", std::vector<std::string>{"KSpi0"}},
    {"KSpipi", std::vector<std::string>{"pipipipi"}},
    {"KSeta", std::vector<std::string>{"pipieta"}},
    {"KSetaPrimerhogamma", std::vector<std::string>{"KSpipipi0"}},
    {"KSpi0pi0", std::vector<std::string>{"pipipi0pi0", "KSKS", "KSpi0gamma"}}
};

### Start calculating the background to signal bin efficiencies, times the ratio of branching fractions

In [6]:
std::string BackgroundToSignalRatios;
for(const auto &Tag : Tags) {
    for(std::size_t i = 0; i < Tag.second.size(); i++) {
        const auto SignalBF = GetBranchingFraction(Tag.first);
        uncertainties::udouble SignalBF_unc(SignalBF.first, SignalBF.second);
        const auto BackgroundBF = GetBranchingFraction(Tag.second[i]);
        uncertainties::udouble BackgroundBF_unc(BackgroundBF.first, BackgroundBF.second);
        const auto BFRatio = BackgroundBF_unc/SignalBF_unc;
        const auto BkgEff_unc = GetRecBackgroundEff(Tag.first, Tag.second[i]);
        const auto SignalEff = GetSTEfficiency(Tag.first);
        uncertainties::udouble SigEff_unc(SignalEff.first, SignalEff.second);
        const auto EffRatio = BkgEff_unc/SigEff_unc;
        const auto BkgToSigRatio = EffRatio*BFRatio;
        std::string Label = Tag.first + "_PeakingBackground" + std::to_string(i);
        Label += "_BackgroundToSignalRatio";
        BackgroundToSignalRatios += Label + " ";
        BackgroundToSignalRatios += std::to_string(uncertainties::nom(BkgToSigRatio)) + "\n";
        BackgroundToSignalRatios += Label + "_err ";
        BackgroundToSignalRatios += std::to_string(uncertainties::sdev(BkgToSigRatio)) + "\n";
    }
}
std::cout << BackgroundToSignalRatios;

KKpipi_PeakingBackground0_BackgroundToSignalRatio 0.011654
KKpipi_PeakingBackground0_BackgroundToSignalRatio_err 0.001078
KSeta_PeakingBackground0_BackgroundToSignalRatio 0.000271
KSeta_PeakingBackground0_BackgroundToSignalRatio_err 0.000032
KSetaPrimerhogamma_PeakingBackground0_BackgroundToSignalRatio 0.030751
KSetaPrimerhogamma_PeakingBackground0_BackgroundToSignalRatio_err 0.004553
KSpi0_PeakingBackground0_BackgroundToSignalRatio 0.002628
KSpi0_PeakingBackground0_BackgroundToSignalRatio_err 0.000482
KSpi0pi0_PeakingBackground0_BackgroundToSignalRatio 0.004386
KSpi0pi0_PeakingBackground0_BackgroundToSignalRatio_err 0.000730
KSpi0pi0_PeakingBackground1_BackgroundToSignalRatio 0.004848
KSpi0pi0_PeakingBackground1_BackgroundToSignalRatio_err 0.000585
KSpi0pi0_PeakingBackground2_BackgroundToSignalRatio 0.000429
KSpi0pi0_PeakingBackground2_BackgroundToSignalRatio_err 0.000060
KSpipi_PeakingBackground0_BackgroundToSignalRatio 0.002352
KSpipi_PeakingBackground0_BackgroundToSignalRatio_err 0

### Save parameters to a file

In [7]:
std::ofstream File("BackgroundToSignalRatios_ST.txt");
File << BackgroundToSignalRatios;
File.close();